# GeoZones mapping — comparison

Compares `geozones-mapping-output.csv` (actual geometries, IoU ≥ 0.4) with
`geozones-mapping-bboxes-output.csv` (bbox matching, IoU ≥ 0.9).

Samples from each diff group and shows matches on a map.

In [1]:
import pandas as pd
import geopandas as gpd
import json, folium
from shapely.geometry import shape

ORIG_CSV      = "./geozones-mapping-output.csv"
BBOX_CSV      = "./geozones-mapping-bboxes-output.csv"
DATASET_CSV   = "./export-dataset.csv"
ZONES_GEOJSON = "./zones_filtered.geojson"
SAMPLE_N      = 10

orig = pd.read_csv(ORIG_CSV)[["id", "zone_id", "iou_score"]].rename(columns={"zone_id": "zone_orig", "iou_score": "iou_orig"})
bbox = pd.read_csv(BBOX_CSV)[["id", "zone_id", "iou_score"]].rename(columns={"zone_id": "zone_bbox", "iou_score": "iou_bbox"})

merged = orig.merge(bbox, on="id", how="outer")

only_orig = merged[merged["zone_bbox"].isna()]
only_bbox = merged[merged["zone_orig"].isna()]
different = merged[merged["zone_orig"].notna() & merged["zone_bbox"].notna() & (merged["zone_orig"] != merged["zone_bbox"])]
same      = merged[merged["zone_orig"].notna() & merged["zone_bbox"].notna() & (merged["zone_orig"] == merged["zone_bbox"])]

print(f"Same zone matched:              {len(same):,}")
print(f"Different zone matched:         {len(different):,}")
print(f"Only in original (actual geom): {len(only_orig):,}")
print(f"Only in bbox:                   {len(only_bbox):,}")

Same zone matched:              27,792
Different zone matched:         85
Only in original (actual geom): 4,468
Only in bbox:                   1,342


## Findings

**Only in original (4,468)** — the actual-geometry approach matched these but bbox didn't.
Looking at the samples, most are regional or departmental datasets whose bbox only partially
overlaps the zone geometry (e.g. IoU ~0.5 against the real shape). These are approximative
matches — losing them with the bbox approach is acceptable since they weren't precise to begin with.

**Only in bbox (1,342)** — bbox matched at ≥ 0.9 but actual geometry IoU was < 0.4.
These turn out to be very precise local datasets (PPR/PPRN, orthophotography, servitudes) whose
bbox aligns perfectly with a commune or département bbox. The actual-geometry approach missed
them because the irregular zone shape (crenelated boundaries, coastal zones) dragged IoU below
the threshold despite the dataset being clearly scoped to that zone. **These are genuinely good
catches** that the bbox approach excels at.

**Different zone (85)** — two recurring patterns:
- DOM-TOM region/département ambiguity: Martinique, Guyane, Réunion each have both a région
  and a département with the same geography — the two approaches just pick different levels.
- EPCI vs commune: the dataset bbox matches a commune more tightly than the EPCI, or vice versa.
  The bbox approach tends to pick the more granular (and often more accurate) level.

In [2]:
## Load zone geometries and dataset geometries for map display

zones_geom = (
    gpd.read_file(ZONES_GEOJSON, engine="pyogrio")[["id", "geometry"]]
    .drop_duplicates("id")
    .set_index("id")["geometry"]
)

df = pd.read_csv(DATASET_CSV, sep=";", dtype=str)

def parse_geom(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    try:
        return shape(json.loads(raw))
    except Exception:
        return None

df["geometry"] = df["spatial.geom"].apply(parse_geom)
dataset_info = df[df["geometry"].notna()].set_index("id")[["title", "geometry"]]

print(f"Zone geometries: {len(zones_geom):,}  |  Dataset geometries: {len(dataset_info):,}")

Zone geometries: 38,340  |  Dataset geometries: 45,431


In [3]:
def show_map(dataset_id, zone_orig=None, zone_bbox=None):
    if dataset_id not in dataset_info.index:
        return
    row = dataset_info.loc[dataset_id]
    dgeom = row["geometry"]
    cx, cy = dgeom.centroid.x, dgeom.centroid.y
    m = folium.Map(location=[cy, cx], zoom_start=7, tiles="CartoDB positron")

    folium.GeoJson(dgeom.__geo_interface__,
        style_function=lambda _: {"color": "red", "fillColor": "red", "fillOpacity": 0.1, "weight": 2},
        tooltip="dataset").add_to(m)

    if zone_orig and (g := zones_geom.get(zone_orig)) is not None:
        folium.GeoJson(g.__geo_interface__,
            style_function=lambda _: {"color": "blue", "fillColor": "blue", "fillOpacity": 0.1, "weight": 2},
            tooltip=f"orig: {zone_orig}").add_to(m)

    if zone_bbox and zone_bbox != zone_orig and (g := zones_geom.get(zone_bbox)) is not None:
        folium.GeoJson(g.__geo_interface__,
            style_function=lambda _: {"color": "green", "fillColor": "green", "fillOpacity": 0.1, "weight": 2},
            tooltip=f"bbox: {zone_bbox}").add_to(m)
    display(m)

def show_sample(group, label, zone_orig_col=None, zone_bbox_col=None):
    sample = group.sample(min(SAMPLE_N, len(group)), random_state=42)
    print(f"\n=== {label} ({len(group):,} total) ===")
    for _, row in sample.iterrows():
        title = dataset_info.loc[row["id"], "title"] if row["id"] in dataset_info.index else "?"
        zo = row.get(zone_orig_col) if zone_orig_col else None
        zb = row.get(zone_bbox_col) if zone_bbox_col else None
        print(f"  orig={zo}  bbox={zb}  —  {str(title)[:80]}")
        show_map(row["id"], zone_orig=zo, zone_bbox=zb)

In [4]:
## Matched by original only (bbox approach missed these)
show_sample(only_orig, "Only in original", zone_orig_col="zone_orig")


=== Only in original (4,468 total) ===
  orig=fr:region:32  bbox=None  —  Le potentiel financier agrégé (PFIA) par habitant rapporté à la moyenne national


  orig=fr:commune:69271  bbox=None  —  Lieux remarquables de la commune de Chassieu


  orig=fr:region:32  bbox=None  —  Parcs naturels


  orig=fr:departement:01  bbox=None  —  RD gabarit


  orig=fr:region:32  bbox=None  —  Les comités locaux d'aide aux projets dans la Région Hauts-de-France - Versant N


  orig=fr:region:32  bbox=None  —  Partenaires du pôle de compétitivité i-Trans


  orig=fr:region:32  bbox=None  —  Nombre de 15-29 ans et part dans la population à l’horizon 2040 par EPCI regroup


  orig=fr:epci:246800726  bbox=None  —  DONNEES : Zonage d'assainissement sur Colmar Agglomération


  orig=fr:region:32  bbox=None  —  Panorama de la gestion de l’eau par bassin versant en région Hauts-de-France


  orig=fr:region:32  bbox=None  —  GAL du Pays du Montreuillois


In [5]:
## Matched by bbox only (original approach missed these)
show_sample(only_bbox, "Only in bbox", zone_bbox_col="zone_bbox")


=== Only in bbox (1,342 total) ===
  orig=None  bbox=fr:commune:64006  —  PPR ACCOUS (64DDTM19970017) - Plan de Prévention des risques naturels (PPRN) de 


  orig=None  bbox=fr:departement:54  —  SUP EL3 - Servitude halage et marchepied en Meurthe-et-Moselle


  orig=None  bbox=fr:departement:54  —  SUP PM1 - Plans de prévention des risques naturels prévisibles et plans de préve


  orig=None  bbox=fr:departement:54  —  DONNEES: Orthophotographie 2010 haute résolution RVB (30cm) sur le département d


  orig=None  bbox=fr:commune:26320  —  PLU de SAINT NAZAIRE EN ROYANS 26320 - Mise à jour 25/01/2018 - FIN de VALIDITE 


  orig=None  bbox=fr:commune:38078  —  PPRN de La-Chapelle-du-Bard approuvé le 31/12/2004


  orig=None  bbox=country-group:world  —  Aires marines protégées françaises


  orig=None  bbox=fr:commune:83136  —  Aléa feu de forêt sur la commune du Thoronet


  orig=None  bbox=fr:commune:64006  —  PPR ACCOUS (révision) (64DDTM20060008) - Zone réglementée du Plan de Prévention 


  orig=None  bbox=fr:departement:31  —  Bruit - Dépassement LN62 Type C (VRU)


In [6]:
## Matched by both but to different zones (blue=original, green=bbox)
show_sample(different, "Different zone matched", zone_orig_col="zone_orig", zone_bbox_col="zone_bbox")


=== Different zone matched (85 total) ===
  orig=fr:epci:243000593  bbox=fr:commune:30341  —  PPRI de Vauvert(30) - périmètre


  orig=fr:departement:972  bbox=fr:region:02  —  Masses d'eau cours d'eau - Martinique - Version Rapportage 2010


  orig=fr:epci:243000593  bbox=fr:commune:30341  —  PPRI de Vauvert(30) - zones d'aléa


  orig=fr:departement:973  bbox=fr:region:03  —  Points d'eau isolés - Guyane 2015 - BD Carthage


  orig=fr:departement:973  bbox=fr:region:03  —  Masses d'eau souterraines - Guyane - Version Rapportage 2016


  orig=fr:epci:243000650  bbox=fr:commune:30276  —  PPRI de Saint-Laurent-d'Aigouze (30) - zonage règlementaire


  orig=fr:region:02  bbox=fr:departement:972  —  Masses d'eau souterraines - Martinique - Version Etat des Lieux 2013


  orig=fr:departement:974  bbox=fr:region:04  —  Polygones élémentaires de masses d'eau souterraines - Réunion - Version Rapporta


  orig=fr:region:03  bbox=fr:departement:973  —  Masses d'eau souterraines - Guyane - Version Rapportage 2010


  orig=fr:epci:243000593  bbox=fr:commune:30341  —  PPRI de Vauvert(30) - zonage règlementaire


## Bbox borderline matches (0.8–0.9)

Samples from `geozones-mapping-bboxes-output.csv` where IoU is between 0.8 and 0.9 —
matches that pass the 0.8 threshold but not 0.9. Helps judge whether 0.8 or 0.9 is the right cutoff.

In [7]:
borderline = pd.read_csv(BBOX_CSV)
borderline = borderline[(borderline["iou_score"] >= 0.8) & (borderline["iou_score"] < 0.9)]
print(f"Bbox matches with 0.8 ≤ IoU < 0.9: {len(borderline):,}")

sample = borderline.sample(min(SAMPLE_N, len(borderline)), random_state=42)
for _, row in sample.iterrows():
    title = dataset_info.loc[row["id"], "title"] if row["id"] in dataset_info.index else "?"
    print(f"  [IoU={row['iou_score']:.3f}] {row['zone_id']}  —  {str(title)[:80]}")
    show_map(row["id"], zone_orig=row["zone_id"])

Bbox matches with 0.8 ≤ IoU < 0.9: 2,509
  [IoU=0.839] fr:epci:200046977  —  Périmètre de polarités commerciales du PLU-H de la Métropole de Lyon


  [IoU=0.894] country-subset:fr:metro  —  DONNEE : Périmètre des départements de France métropolitaine


  [IoU=0.836] fr:commune:30167  —  PPRI de Meyrannes (30) - périmètre


  [IoU=0.897] fr:commune:64054  —  PPR ARROS-DE-NAY (64DDTM20010002) - Zone réglementée du Plan de Prévention du Ri


  [IoU=0.894] country-subset:fr:metro  —  DONNEE : Centroïde des régions de France métropolitaine


  [IoU=0.869] fr:commune:30254  —  PPRI de Saint-Geniès-de-Comolas (30) - zoange règlementaire


  [IoU=0.879] fr:commune:54277  —  PPRN – Lot de données géographiques du PPR Inondation - JEANDELIZE (54277)


  [IoU=0.852] fr:commune:64189  —  PPR CIBOURE (64DDTM19870004) - Périmètre du Plan de Prévention du Risque Inondat


  [IoU=0.877] fr:departement:17  —  Zone réglementée du PPRN inondation par débordement du secteur médian la Boutonn


  [IoU=0.815] country-subset:fr:metro  —  Masses d'eau plan d'eau - Métropole - Version État des Lieux 2019
